In [ ]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep


In [ ]:
sampling_rate = 20 #Hz
freq_list = np.array([0.1 , 0.12, 0.14, 0.16, 0.18, 0.2 , 0.22, 0.24, 0.26, 0.28, 0.3, 0.32, 0.34, 0.36, 0.38, 0.4])

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int)


A = 5
samples = 50
repeat = 200

interval = 50e-3
exposure_time = 500e-6 #500 us

measurement = 'SPADE'

In [ ]:
from dataclasses import dataclass
import os

class _Save:
    def save(self, file: str):
        file = os.path.expanduser(file)
        if not os.path.exists(file):
            np.savez_compressed(file, **self.__dict__)
        else:
            print(f'ERROR: File {file} already exists.')


@dataclass
class Captured(_Save):
    measurement: str
    real_f: float

    raw: np.ndarray
    timestamp: np.ndarray




@dataclass
class Estimates(_Save):
    measurement: str
    real_f: float

    cropped: np.ndarray
    td: np.ndarray
    freq: np.ndarray
    pn: float


In [ ]:
with qCMOS() as qcmos:
    with DMD() as dmd:
        for i, picture_time in tqdm(enumerate(pic_time)):

            dmd.ez_load_seq([dmd.ez_single_pixel(0), dmd.ez_single_pixel(A)], picture_time)
            qcmos.ez_exposure_time(exposure_time)
            qcmos.ez_triggersource_masterpluse(samples, interval)

            if measurement.upper() == 'SPADE':
                qcmos.ez_roi(**SPADE.ROI)
            elif measurement.upper() == 'DI':
                qcmos.ez_roi(**DI.ROI)

            raw, timestamp = [], []
            for _ in tqdm(range(repeat)):
                qcmos.buf_alloc(samples)
                qcmos.cap_snapshot()

                dmd.Run()
                sleep(1e-6)
                qcmos.cap_firetrigger()

                qcmos.ez_wait_capture()

                qcmos.cap_stop()
                dmd.Halt()

                raw_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = qcmos.ez_read_buf(frame)
                    raw_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                qcmos.buf_release()

                raw.append(raw_)
                timestamp.append(timestamp_)


            cap = Captured(measurement.upper(), freq_list[i], raw, timestamp)
            cap.save(f'./__temp__/{measurement.lower()}_{freq_list[i]}.npz')
